### ***GOLD: DAILY STOCK PERFORMANCE***

In [3]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

catalog_db = "FABRIC_COE_WORKLOAD.stock_market_analysis_LH"

silver_table = f"{catalog_db}.silver.slv_stock_prices"
gold_performance = f"{catalog_db}.gold.gld_daily_stock_performance"


# 1. GOLD: DAILY STOCK PERFORMANCE


df_silver = (
    spark.table(silver_table)
    .filter(col("is_current") == True)
    .filter(col("ticker").isNotNull())
    .filter(col("close").isNotNull() & (col("close") > 0))
    .withColumn("trade_date", to_date("trade_datetime"))
    .filter(col("trade_date").isNotNull())
)

df_daily = (
    df_silver
    .groupBy("ticker", "currency", "trade_date")
    .agg(
        min_by("open", "trade_datetime").alias("open"),
        max("high").alias("high"),
        min("low").alias("low"),
        max_by("close", "trade_datetime").alias("close"),
        sum("volume").alias("total_volume"),
        max_by("adj_close", "trade_datetime").alias("adj_close")
    )
)

w = Window.partitionBy("ticker").orderBy("trade_date")

df_performance = (
    df_daily
    .withColumn("previous_close", lag("close").over(w))
    .withColumn(
        "daily_return_pct",
        when(
            col("previous_close") > 0,
            (col("close") - col("previous_close"))
            / col("previous_close") * 100
        )
    )
    .withColumn("moving_avg_7", avg("close").over(w.rowsBetween(-6, 0)))
    .withColumn("moving_avg_30", avg("close").over(w.rowsBetween(-29, 0)))
    .withColumn(
        "volatility_30",
        stddev_samp("daily_return_pct").over(w.rowsBetween(-29, 0))
    )
    .withColumn(
        "cumulative_return_pct",
        (
            col("close")
            / first("close").over(
                w.rowsBetween(Window.unboundedPreceding, 0)
            ) - 1
        ) * 100
    )
    .withColumn("processed_at", current_timestamp())
)

if not spark.catalog.tableExists(gold_performance):

    df_performance.write\
        .format("delta")\
        .mode("errorifexists")\
        .option("mergeSchema", True)\
        .saveAsTable(gold_performance)

    print("gld_daily_stock_performance Delta table created successfully")

else:

    print("gld_daily_stock_performance Delta table Already Exists performing SCD Type 1 logic ")

    df_performance.createOrReplaceTempView("vw_gold_performance_source")

    spark.sql(f"""
        MERGE INTO {gold_performance} AS target
        USING vw_gold_performance_source AS source
        ON target.ticker = source.ticker
        AND target.trade_date = source.trade_date

        WHEN MATCHED THEN UPDATE SET *

        WHEN NOT MATCHED THEN INSERT *
    """)

    print("gld_daily_stock_performance Delta table merged successfully")

display(spark.table(gold_performance))


StatementMeta(, dfced91c-c37e-4cc2-93a7-ee961913cae4, 5, Finished, Available, Finished, False)

gld_daily_stock_performance Delta table Already Exists performing SCD Type 1 logic 
gld_daily_stock_performance Delta table merged successfully


SynapseWidget(Synapse.DataFrame, 4d26ab00-cc09-45fe-a7ce-8d277eacb102)

### ***GOLD: PRICE-BASED SENTIMENT ANALYSIS***

In [5]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

catalog_db = "FABRIC_COE_WORKLOAD.stock_market_analysis_LH"

silver_table = f"{catalog_db}.silver.slv_stock_prices"
gold_sentiment = f"{catalog_db}.gold.gld_stock_sentiment_analysis"

# 2. GOLD: PRICE-BASED SENTIMENT ANALYSIS


df_labeled = (
    spark.table(gold_performance)
    .withColumn(
        "sentiment_score",
        when(col("daily_return_pct") > 0, 1.0)
        .when(col("daily_return_pct") < 0, -1.0)
        .otherwise(0.0)
    )
    .withColumn(
        "sentiment_label",
        when(col("daily_return_pct") > 0, "Positive")
        .when(col("daily_return_pct") < 0, "Negative")
        .otherwise("Neutral")
    )
)

w_all = Window.partitionBy("ticker")

df_sentiment = (
    df_labeled
    .withColumn(
        "trading_days",
        count("daily_return_pct").over(w_all)
    )
    .withColumn(
        "positive_days",
        sum(when(col("sentiment_label") == "Positive", 1).otherwise(0))
        .over(w_all)
    )
    .withColumn(
        "negative_days",
        sum(when(col("sentiment_label") == "Negative", 1).otherwise(0))
        .over(w_all)
    )
    .withColumn(
        "neutral_days",
        sum(when(col("sentiment_label") == "Neutral", 1).otherwise(0))
        .over(w_all)
    )
    .withColumn(
        "overall_sentiment_score",
        avg("sentiment_score").over(w_all)
    )
    .withColumn(
        "overall_sentiment",
        when(col("overall_sentiment_score") > 0, "Positive")
        .when(col("overall_sentiment_score") < 0, "Negative")
        .otherwise("Neutral")
    )
    .withColumn("processed_at", current_timestamp())
    .select(
        "ticker", "currency", "trade_date", "close",
        "daily_return_pct", "sentiment_score", "sentiment_label",
        "trading_days", "positive_days", "negative_days",
        "neutral_days", "overall_sentiment_score",
        "overall_sentiment", "processed_at"
    )
)

if not spark.catalog.tableExists(gold_sentiment):
    
    df_sentiment.write\
        .format("delta")\
        .mode("errorifexists")\
        .option("mergeSchema", True)\
        .saveAsTable(gold_sentiment)

    print(" gld_stock_sentiment_analysis Delta table created successfully")
    
else:

    print(" gld_stock_sentiment_analysis Delta table Already Exists performing SCD Type 1 logic")

    df_sentiment.createOrReplaceTempView("vw_gold_sentiment_source")

    spark.sql(f"""
        MERGE INTO {gold_sentiment} AS target
        USING vw_gold_sentiment_source AS source
        ON target.ticker = source.ticker
        AND target.trade_date = source.trade_date

        WHEN MATCHED THEN UPDATE SET *

        WHEN NOT MATCHED THEN INSERT *
    """)
    print(" gld_stock_sentiment_analysis Delta table created successfully")

display(spark.table(gold_sentiment))


StatementMeta(, dfced91c-c37e-4cc2-93a7-ee961913cae4, 7, Finished, Available, Finished, False)

 gld_stock_sentiment_analysis Delta table Already Exists performing SCD Type 1 logic
 gld_stock_sentiment_analysis Delta table created successfully


SynapseWidget(Synapse.DataFrame, 2fa07ba5-166d-48e7-a7de-0e98c2352685)

### ***GOLD: SIMPLE 7-TRADING-DAY BASELINE FORECAST***

In [7]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

catalog_db = "FABRIC_COE_WORKLOAD.stock_market_analysis_LH"

silver_table = f"{catalog_db}.silver.slv_stock_prices"
gold_forecast = f"{catalog_db}.gold.gld_stock_price_forecast"

# 3. GOLD: SIMPLE 7-TRADING-DAY BASELINE FORECAST


w_latest = Window.partitionBy("ticker").orderBy(col("trade_date").desc())

df_latest = (
    spark.table(gold_performance)
    .withColumn("rn", row_number().over(w_latest))
    .filter(col("rn") == 1)
    .select(
        "ticker", "currency",
        col("trade_date").alias("last_actual_date"),
        col("close").alias("last_actual_close")
    )
)

df_forecast = (
    df_latest
    .withColumn(
        "forecast_date",
        explode(sequence(
            date_add("last_actual_date", 1),
            date_add("last_actual_date", 14)
        ))
    )
    .filter(dayofweek("forecast_date").between(2, 6))
    .withColumn(
        "forecast_number",
        row_number().over(
            Window.partitionBy("ticker").orderBy("forecast_date")
        )
    )
    .filter(col("forecast_number") <= 7)
    .withColumn("predicted_close", col("last_actual_close"))
    .withColumn("forecast_method", lit("Last-close baseline"))
    .withColumn("processed_at", current_timestamp())
    .select(
        "ticker", "currency", "last_actual_date",
        "last_actual_close", "forecast_date",
        "forecast_number", "predicted_close",
        "forecast_method", "processed_at"
    )
)

if not spark.catalog.tableExists(gold_forecast):

    df_forecast.write\
        .format("delta")\
        .mode("errorifexists")\
        .option("mergeSchema", True)\
        .saveAsTable(gold_forecast)

    print("gld_stock_price_forecast Delta table created successfully")

else:

    print("gld_stock_price_forecast Delta table Already Exists performing SCD Type 1 logic")

    df_forecast.createOrReplaceTempView("vw_gold_forecast_source")

    spark.sql(f"""
        MERGE INTO {gold_forecast} AS target
        USING vw_gold_forecast_source AS source
        ON target.ticker = source.ticker
        AND target.forecast_date = source.forecast_date

        WHEN MATCHED THEN UPDATE SET *

        WHEN NOT MATCHED THEN INSERT *
    """)

    print("gld_stock_price_forecast Delta table merged successfully")

display(spark.table(gold_forecast))

StatementMeta(, dfced91c-c37e-4cc2-93a7-ee961913cae4, 9, Finished, Available, Finished, False)

gld_stock_price_forecast Delta table Already Exists performing SCD Type 1 logic
gld_stock_price_forecast Delta table merged successfully


SynapseWidget(Synapse.DataFrame, ee2b4b44-65e5-4b48-a87d-871a26bb6a4b)